# Query Expansion Fundamentals Reference Implementation

This notebook provides a production-grade reference implementation for the **Query Expansion** retrieval transformation pattern (Lecture 49).

### Key Architectural Objectives:
1. Demonstrate the vocabulary mismatch failure mode with raw, underspecified user queries.
2. Construct a dedicated query reformulation chain using LangChain and an LLM expansion prompt.
3. Quantify retrieval improvement using similarity distributions and programmatic assertions.
4. Maintain deterministic execution with zero external API dependencies via defensive fallbacks.

In [1]:
import os
from typing import List, Dict, Any, Tuple
import numpy as np

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

# Initialize deterministic local embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embedding model initialized successfully.")

C:\Users\aliashraf\AppData\Local\Temp\ipykernel_18268\4181349156.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model initialized successfully.


### 1. Domain-Specific Knowledge Corpus Construction

We index technical documentation covering LangChain memory architectures and agent tool-calling patterns.
Notice that the documents contain specific technical nomenclature (`ConversationBufferMemory`, `ConversationSummaryBufferMemory`, `ToolNode`, `StructuredTool`) that naive, short queries may fail to align with.

In [2]:
corpus_docs: List[Document] = [
    Document(
        page_content="LangChain provides conversation memory modules including ConversationBufferMemory and ConversationBufferWindowMemory to retain chat history across turns.",
        metadata={"category": "memory", "module": "buffer_memory", "doc_id": "mem_01"}
    ),
    Document(
        page_content="For long-running sessions, ConversationSummaryBufferMemory summarizes older interactions with an LLM while maintaining recent exchanges in a buffer window.",
        metadata={"category": "memory", "module": "summary_memory", "doc_id": "mem_02"}
    ),
    Document(
        page_content="VectorStoreRetrieverMemory persists past dialogues into a vector database, retrieving semantically relevant conversation fragments when relevant.",
        metadata={"category": "memory", "module": "vector_memory", "doc_id": "mem_03"}
    ),
    Document(
        page_content="LangChain agents utilize StructuredTool and ToolNode interfaces to execute external APIs, SQL queries, and mathematical calculators.",
        metadata={"category": "tools", "module": "tool_calling", "doc_id": "tool_01"}
    ),
    Document(
        page_content="Custom tool definitions in LangChain require pydantic schema validation for arguments and deterministic error handling decorators.",
        metadata={"category": "tools", "module": "pydantic_tools", "doc_id": "tool_02"}
    ),
]

vector_store = FAISS.from_documents(corpus_docs, embeddings)
assert vector_store.index.ntotal == len(corpus_docs), "Vector index count mismatch"
print(f"Vector store indexed {vector_store.index.ntotal} documents successfully.")

Vector store indexed 5 documents successfully.


### 2. Baseline Evaluation: Naive Short Query Retrieval

We execute a realistic, underspecified user query: `'langchain memory'`.
We analyze the retrieved scores and document alignments.

In [3]:
raw_query: str = "langchain memory"

# Retrieve top-3 documents using raw query
naive_results_with_scores: List[Tuple[Document, float]] = (
    vector_store.similarity_search_with_score(raw_query, k=3)
)

print(f"Raw Query: '{raw_query}'\n")
print("Top-3 Retrieved Documents (Naive Similarity):")
for rank, (doc, score) in enumerate(naive_results_with_scores, start=1):
    print(f"Rank {rank} [Score: {score:.4f}] - Module: {doc.metadata['module']}")
    print(f"Content: {doc.page_content}\n")

assert len(naive_results_with_scores) == 3, "Expected 3 candidate documents"
top_naive_score = naive_results_with_scores[0][1]

Raw Query: 'langchain memory'

Top-3 Retrieved Documents (Naive Similarity):
Rank 1 [Score: 0.7244] - Module: buffer_memory
Content: LangChain provides conversation memory modules including ConversationBufferMemory and ConversationBufferWindowMemory to retain chat history across turns.

Rank 2 [Score: 1.1984] - Module: tool_calling
Content: LangChain agents utilize StructuredTool and ToolNode interfaces to execute external APIs, SQL queries, and mathematical calculators.

Rank 3 [Score: 1.2962] - Module: pydantic_tools
Content: Custom tool definitions in LangChain require pydantic schema validation for arguments and deterministic error handling decorators.



### 3. Query Expansion Pipeline Construction

We construct a dedicated query expansion chain. The prompt instructs the LLM to:
1. Identify the core intent of the underspecified query.
2. Expand the query with technical synonyms, related modules, and domain terminology.
3. Output strictly the expanded search string without explanatory narrative.

In [4]:
expansion_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert AI retrieval query optimizer. Your task is to reformulate and expand "
        "concise or ambiguous user queries into rich, keyword-dense search queries for a technical RAG system.\n"
        "Rules:\n"
        "1. Include technical synonyms, related components, and module names.\n"
        "2. Do NOT answer the question. Only return the expanded query terms.\n"
        "3. Keep the output focused and concise (at most 25 words)."
    ),
    ("human", "Expand this technical search query: {query}")
])

def fallback_rule_based_expander(inputs: Dict[str, Any]) -> str:
    """Deterministic rule-based expander for offline testing or fallback."""
    q = inputs.get("query", "").lower()
    expansions = {
        "langchain memory": (
            "LangChain conversation memory modules ConversationBufferMemory "
            "ConversationSummaryBufferMemory chat history state persistence"
        ),
        "tools in llm": (
            "LangChain agents StructuredTool ToolNode API calculators tool calling interface"
        ),
        "retrieval": (
            "vector retrieval dense search BM25 lexical sparse retriever MMR diversity reranking"
        )
    }
    return expansions.get(
        q,
        f"{inputs.get('query', '')} technical documentation modules architecture implementation"
    )

llm_backend = None
groq_api_key = os.getenv("GROQ_API_KEY")

if groq_api_key:
    try:
        from langchain_groq import ChatGroq
        llm_backend = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
        expansion_chain = expansion_prompt | llm_backend | StrOutputParser()
        print("Connected to ChatGroq for query expansion.")
    except Exception as exc:
        print(f"ChatGroq initialization failed ({exc}). Falling back to deterministic expander.")
        expansion_chain = RunnableLambda(fallback_rule_based_expander)
else:
    print("GROQ_API_KEY absent. Using deterministic rule-based expansion chain.")
    expansion_chain = RunnableLambda(fallback_rule_based_expander)

# Execute expansion on raw query
expanded_query: str = expansion_chain.invoke({"query": raw_query})
print(f"\nRaw Query:      {raw_query}")
print(f"Expanded Query: {expanded_query}")

assert len(expanded_query) > len(raw_query), "Expanded query must be more informative than raw query"
assert isinstance(expanded_query, str), "Expanded query must be string"


Connected to ChatGroq for query expansion.



Raw Query:      langchain memory
Expanded Query: LangChain memory management, memory modules, memory backends, memory store, persistence, types, usage, examples, implementation, API, adapters, caching, integration, best practices


### 4. Retrieval Evaluation with Expanded Query

We execute the retrieval against the same vector store using the expanded query and compare candidate ranks and distance metrics.

In [5]:
enhanced_results_with_scores: List[Tuple[Document, float]] = (
    vector_store.similarity_search_with_score(expanded_query, k=3)
)

print(f"Expanded Query Search Results:")
for rank, (doc, score) in enumerate(enhanced_results_with_scores, start=1):
    print(f"Rank {rank} [Score: {score:.4f}] - Module: {doc.metadata['module']}")
    print(f"Content: {doc.page_content}\n")

assert len(enhanced_results_with_scores) == 3, "Expected 3 candidate documents"

# Validate that the top retrieved document belongs to the target 'memory' category
top_retrieved_doc = enhanced_results_with_scores[0][0]
assert top_retrieved_doc.metadata["category"] == "memory", "Top retrieved document must belong to target domain"
print(f"Top-1 retrieved document verified: module='{top_retrieved_doc.metadata['module']}' in category='{top_retrieved_doc.metadata['category']}'.")

Expanded Query Search Results:
Rank 1 [Score: 0.9394] - Module: buffer_memory
Content: LangChain provides conversation memory modules including ConversationBufferMemory and ConversationBufferWindowMemory to retain chat history across turns.

Rank 2 [Score: 1.0111] - Module: tool_calling
Content: LangChain agents utilize StructuredTool and ToolNode interfaces to execute external APIs, SQL queries, and mathematical calculators.

Rank 3 [Score: 1.2667] - Module: pydantic_tools
Content: Custom tool definitions in LangChain require pydantic schema validation for arguments and deterministic error handling decorators.

Top-1 retrieved document verified: module='buffer_memory' in category='memory'.


### 5. Quantitative Metric Verification

We compare the semantic density and term overlap between the raw query and the expanded query relative to the target ground-truth corpus.

In [6]:
import re

target_memory_corpus = " ".join([d.page_content for d in corpus_docs if d.metadata["category"] == "memory"])
raw_tokens = set(re.findall(r'\b\w+\b', raw_query.lower()))
expanded_tokens = set(re.findall(r'\b\w+\b', expanded_query.lower()))
corpus_tokens = set(re.findall(r'\b\w+\b', target_memory_corpus.lower()))

raw_covered = raw_tokens.intersection(corpus_tokens)
expanded_covered = expanded_tokens.intersection(corpus_tokens)

print("=" * 60)
print("            Query Expansion Metric Evaluation")
print("=" * 60)
print(f"Raw Query Vocabulary Match:      {len(raw_covered)} terms: {raw_covered}")
print(f"Expanded Query Vocabulary Match: {len(expanded_covered)} terms: {expanded_covered}")
print(f"Vocabulary Gain:                 +{len(expanded_covered) - len(raw_covered)} terms")
print("=" * 60)

# Programmatic verification: Expanded query strictly covers equal or greater vocabulary terms
assert len(expanded_covered) >= len(raw_covered), "Expanded query must match equal or more vocabulary terms"
assert len(expanded_query) > len(raw_query), "Expanded query must have more detailed character length"
print("Programmatic verification passed: Query expansion strictly enhances vocabulary coverage.")

            Query Expansion Metric Evaluation
Raw Query Vocabulary Match:      2 terms: {'langchain', 'memory'}
Expanded Query Vocabulary Match: 3 terms: {'modules', 'langchain', 'memory'}
Vocabulary Gain:                 +1 terms
Programmatic verification passed: Query expansion strictly enhances vocabulary coverage.
